peers list:

In [2]:
peers = [
    "HDFCBANK.NS",
    "ICICIBANK.NS",
    "AXISBANK.NS",
    "KOTAKBANK.NS",
    "SBIN.NS"
]


Load peer fundamentals (P/B, P/E):
P/B - Primary multiple
P/E - secondary check

In [3]:
import pandas as pd
import numpy as np
import yfinance as yf

rows = []
for ticker in peers:
    info = yf.Ticker(ticker).info or {}
    rows.append({
        "ticker": ticker,
        "price_to_book": info.get("priceToBook"),
        "price_to_earnings": info.get("trailingPE") or info.get("forwardPE"),
        "market_cap": info.get("marketCap")
    })

fundamentals = pd.DataFrame(rows)
fundamentals = fundamentals.dropna(subset=["price_to_book", "price_to_earnings", "market_cap"])
fundamentals


,ticker,price_to_book,price_to_earnings,market_cap
0,HDFCBANK.NS,3.2,18.5,1.400000e+13
1,ICICIBANK.NS,2.6,16.2,9.000000e+12
2,AXISBANK.NS,2.1,14.8,6.000000e+12
3,KOTAKBANK.NS,3.8,22.1,7.000000e+12
4,SBIN.NS,1.6,12.4,4.500000e+13


Cluster peers:

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np

X = fundamentals[["price_to_book", "price_to_earnings", "market_cap"]]
X["market_cap"] = np.log(X["market_cap"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=42)
fundamentals["cluster"] = kmeans.fit_predict(X_scaled)

fundamentals


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22840\2596304374.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["market_cap"] = np.log(X["market_cap"])


,ticker,price_to_book,price_to_earnings,market_cap,cluster
0,HDFCBANK.NS,3.2,18.5,1.400000e+13,0
1,ICICIBANK.NS,2.6,16.2,9.000000e+12,0
2,AXISBANK.NS,2.1,14.8,6.000000e+12,0
3,KOTAKBANK.NS,3.8,22.1,7.000000e+12,0
4,SBIN.NS,1.6,12.4,4.500000e+13,1


Selecting HDFC Bank’s peer cluster:

In [5]:
hdfc_cluster = fundamentals.loc[
    fundamentals["ticker"] == "HDFCBANK.NS", "cluster"
].values[0]

peer_group = fundamentals[fundamentals["cluster"] == hdfc_cluster]
peer_group


,ticker,price_to_book,price_to_earnings,market_cap,cluster
0,HDFCBANK.NS,3.2,18.5,1.400000e+13,0
1,ICICIBANK.NS,2.6,16.2,9.000000e+12,0
2,AXISBANK.NS,2.1,14.8,6.000000e+12,0
3,KOTAKBANK.NS,3.8,22.1,7.000000e+12,0


Computing fair multiples:

In [6]:
pb_median = peer_group["price_to_book"].median()
pe_median = peer_group["price_to_earnings"].median()

pb_median, pe_median


(np.float64(2.9000000000000004), np.float64(17.35))

Implied valuation using P/B:
Load HDFC book value

In [7]:
hdfc_row = fundamentals[fundamentals["ticker"] == "HDFCBANK.NS"].iloc[0]
book_value = hdfc_row["market_cap"] / hdfc_row["price_to_book"]
book_value


Fair value via P/B:

In [8]:
implied_equity_value_pb = pb_median * book_value
implied_equity_value_pb


np.float64(13050000000000.002)

Comparing with DCF & Risk Summary:

In [9]:
dcf_df = pd.read_csv("../data/dcf_hdfc.csv")
risk_df = pd.read_csv("../data/risk_summary_hdfc.csv")

dcf_value = dcf_df.loc[0, "enterprise_value"]
p10 = risk_df[risk_df["metric"] == "P10 (Bear Case)"]["value"].values[0]
p90 = risk_df[risk_df["metric"] == "P90 (Bull Case)"]["value"].values[0]


Final comparison table:

In [10]:
valuation_summary = pd.DataFrame({
    "Method": ["DCF Base", "Monte Carlo P10", "Monte Carlo P90", "P/B Multiple"],
    "Value": [dcf_value, p10, p90, implied_equity_value_pb]
})

valuation_summary


,Method,Value
0,DCF Base,2.019129e+13
1,Monte Carlo P10,1.688332e+13
2,Monte Carlo P90,2.492064e+13
3,P/B Multiple,1.305000e+13


Save outputs:

In [11]:
valuation_summary.to_csv("../data/valuation_triangulation_hdfc.csv", index=False)
peer_group.to_csv("../data/peer_group_hdfc.csv", index=False)


Peer EV/EBITDA data:

In [12]:
import pandas as pd
import numpy as np


In [13]:
import pandas as pd
import numpy as np
import yfinance as yf

peer_rows = []
for ticker in ["HDFCBANK.NS", "ICICIBANK.NS", "AXISBANK.NS", "KOTAKBANK.NS"]:
    info = yf.Ticker(ticker).info or {}
    market_cap = info.get("marketCap")
    total_debt = info.get("totalDebt")
    cash = info.get("totalCash")
    ebitda = info.get("ebitda")

    if None in (market_cap, total_debt, cash, ebitda):
        continue

    peer_rows.append({
        "ticker": ticker,
        "market_cap": market_cap,
        "total_debt": total_debt,
        "cash": cash,
        "ebitda": ebitda
    })

peer_ev_data = pd.DataFrame(peer_rows)
peer_ev_data


,ticker,market_cap,total_debt,cash,ebitda
0,HDFCBANK.NS,1.400000e+13,1.100000e+13,2.500000e+12,2.100000e+12
1,ICICIBANK.NS,9.000000e+12,8.500000e+12,2.000000e+12,1.800000e+12
2,AXISBANK.NS,6.000000e+12,6.800000e+12,1.600000e+12,1.400000e+12
3,KOTAKBANK.NS,7.000000e+12,5.500000e+12,1.800000e+12,1.600000e+12


Compute EV and EV/EBITDA:

In [14]:
peer_ev_data["enterprise_value"] = (
    peer_ev_data["market_cap"]
    + peer_ev_data["total_debt"]
    - peer_ev_data["cash"]
)

peer_ev_data["ev_ebitda"] = (
    peer_ev_data["enterprise_value"] / peer_ev_data["ebitda"]
)

peer_ev_data


,ticker,market_cap,total_debt,cash,ebitda,enterprise_value,ev_ebitda
0,HDFCBANK.NS,1.400000e+13,1.100000e+13,2.500000e+12,2.100000e+12,2.250000e+13,10.714286
1,ICICIBANK.NS,9.000000e+12,8.500000e+12,2.000000e+12,1.800000e+12,1.550000e+13,8.611111
2,AXISBANK.NS,6.000000e+12,6.800000e+12,1.600000e+12,1.400000e+12,1.120000e+13,8.000000
3,KOTAKBANK.NS,7.000000e+12,5.500000e+12,1.800000e+12,1.600000e+12,1.070000e+13,6.687500


Fair EV/EBITDA multiple (median):

In [15]:
ev_ebitda_median = peer_ev_data["ev_ebitda"].median()
ev_ebitda_median


np.float64(8.305555555555555)

Apply EV/EBITDA to HDFC Bank:

In [16]:
hdfc_ebitda = peer_ev_data.loc[
    peer_ev_data["ticker"] == "HDFCBANK.NS", "ebitda"
].iloc[0]

implied_ev = ev_ebitda_median * hdfc_ebitda
implied_ev


np.float64(17441666666666.666)

Convert EV → Equity Value:

In [17]:
hdfc_row = peer_ev_data[peer_ev_data["ticker"] == "HDFCBANK.NS"].iloc[0]
hdfc_debt = hdfc_row["total_debt"]
hdfc_cash = hdfc_row["cash"]

implied_equity_value_ev = implied_ev - hdfc_debt + hdfc_cash
implied_equity_value_ev


np.float64(8941666666666.666)

Update final valuation comparison table:

In [18]:
valuation_summary = pd.DataFrame({
    "Method": [
        "DCF Base",
        "Monte Carlo P10",
        "Monte Carlo P90",
        "P/B Multiple",
        "EV/EBITDA Multiple"
    ],
    "Value": [
        dcf_value,
        p10,
        p90,
        implied_equity_value_pb,
        implied_equity_value_ev
    ]
})

valuation_summary


,Method,Value
0,DCF Base,2.019129e+13
1,Monte Carlo P10,1.688332e+13
2,Monte Carlo P90,2.492064e+13
3,P/B Multiple,1.305000e+13
4,EV/EBITDA Multiple,8.941667e+12


Save outputs (for later stages):

In [19]:
peer_ev_data.to_csv("../data/peer_ev_ebitda.csv", index=False)
valuation_summary.to_csv("../data/valuation_summary.csv", index=False)
